# all-reduce-compose — worked example 2: Build an all_reduce MEAN by composing SUM then dividing

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `all-reduce-compose`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

There is no built-in MEAN reduce op in most backends, so the standard recipe is: do an all_reduce with SUM (= `reduce(SUM, dst=0)` then `broadcast(src=0)`), then divide the broadcast result by `world_size` locally on every rank. Because every rank received the identical global sum, dividing by the same constant leaves every rank with the identical mean.

## Worked solution

**Step 1 — local tensor.** Each rank wraps its value: `tensor = t.tensor([local_value], dtype=t.float32)`.

**Step 2 — reduce SUM to rank 0.** `dist_module.reduce(tensor, dst=0, op=dist_module.ReduceOp.SUM)` accumulates the global sum into rank 0 only.

**Step 3 — broadcast the sum.** `dist_module.broadcast(tensor, src=0)` makes the global sum appear on every rank. At this point we have an all_reduce(SUM) — exactly the ex1 composition.

**Step 4 — local divide for the mean.** `tensor /= world_size` runs independently on every rank. No communication is needed because the broadcast already guaranteed identical inputs; dividing identical values by an identical constant yields identical means. This is why MEAN is implemented as SUM-then-scale rather than a dedicated collective.

**Why it works:** the mean is a linear function of the sum, and the costly part (the global reduction) is shared via the broadcast; the cheap scaling is replicated locally.

In [ ]:
import threading

class MockDist:
    class ReduceOp:
        SUM = 'sum'
        MAX = 'max'
    def __init__(self, world_size):
        self.world_size = world_size
        self._barrier = threading.Barrier(world_size)
        self._slots = [None] * world_size
        self._result = [None]
    def reduce(self, tensor, dst, op):
        rank = threading.current_thread().rank
        self._slots[rank] = tensor.clone()
        self._barrier.wait()
        if rank == dst:
            vals = t.stack(self._slots)
            tensor.copy_(vals.sum(dim=0) if op == self.ReduceOp.SUM else vals.amax(dim=0))
            self._result[0] = tensor.clone()
        self._barrier.wait()
    def broadcast(self, tensor, src):
        self._barrier.wait()
        tensor.copy_(self._result[0])
        self._barrier.wait()

def ex_all_reduce_mean(rank: int, world_size: int, dist_module, local_value: float) -> float:
    tensor = t.tensor([local_value], dtype=t.float32)
    dist_module.reduce(tensor, dst=0, op=dist_module.ReduceOp.SUM)
    dist_module.broadcast(tensor, src=0)
    tensor /= world_size
    return tensor.item()

world_size = 4
locals_ = [4.0, 8.0, 2.0, 6.0]
mock = MockDist(world_size)
results = [None] * world_size

def _run(rank):
    threading.current_thread().rank = rank
    results[rank] = ex_all_reduce_mean(rank, world_size, mock, locals_[rank])

threads = [threading.Thread(target=_run, args=(r,)) for r in range(world_size)]
for th in threads: th.start()
for th in threads: th.join()
expected = sum(locals_) / world_size
print('per-rank means:', results)
print('all equal:', len(set(results)) == 1, '-> mean', results[0], '(expected', expected, ')')